# L.A.Cosmic — Implementation / 구현

**Paper**: van Dokkum, P. G. (2001). "Cosmic-Ray Rejection by Laplacian Edge Detection", *PASP*, 113, 1420. [DOI: 10.1086/323894]

## Overview / 개요

이 노트북은 L.A.Cosmic 알고리즘을 NumPy + SciPy로 직접 구현하고 합성 영상에서 검증한다:
1. 합성 천체영상 (별 + 작은 은하 + 우주선) 생성.
2. 2× 서브샘플링 + Laplacian 컨볼루션 + 양수 부분 보존 + 블록 평균 (Eqs. 5-8).
3. 잡음 모델 (Poisson + read-noise, Eq. 10) 과 유의도 영상 (Eq. 11).
4. Fine-structure image \$\mathcal F\$ (Eq. 14)와 \$\mathcal L^+/\mathcal F\$ 비율로 별 vs CR 구분.
5. 반복 적용으로 큰 CR 제거.
6. 검출 정확도 (TPR / FPR) 측정.

This notebook implements L.A.Cosmic from scratch with NumPy + SciPy and tests on a synthetic field with stars, small galaxies, and injected cosmic rays.


In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from numpy.typing import NDArray

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 10

rng = np.random.default_rng(42)


---

## Part 1: Synthetic field / 합성 영상

별(2D Gaussian PSF, FWHM ≈ 2.4 px) + 작은 은하(extended Gaussian) + 우주선(1~5 pixel sharp spikes).


In [ ]:
def make_synthetic_field(
    size: int = 256,
    n_stars: int = 80,
    n_galaxies: int = 20,
    n_cr: int = 60,
    seed: int = 0,
    sky: float = 100.0,
    gain: float = 1.0,
    rn: float = 5.0,
) -> tuple[np.ndarray, np.ndarray]:
    """Build a synthetic CCD field with stars, galaxies, and cosmic rays.

    Args:
        size: Image side length.
        n_stars: Number of stellar PSFs to inject.
        n_galaxies: Number of extended galaxy profiles.
        n_cr: Number of cosmic-ray hits.
        seed: PRNG seed.
        sky: Sky background in e-.
        gain: e-/ADU.
        rn: Read noise in e-.

    Returns:
        Tuple (image, cr_mask) — both shape (size, size); image in e-, mask boolean.
    """
    rng_local = np.random.default_rng(seed)
    img = np.full((size, size), sky, dtype=np.float64)
    cr_mask = np.zeros((size, size), dtype=bool)

    # Stars (well-sampled, FWHM 2.4 px ~ sigma 1.0)
    sigma_psf = 1.0
    for _ in range(n_stars):
        cx, cy = rng_local.uniform(5, size - 5, 2)
        peak = rng_local.uniform(200, 2000)
        yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
        img += peak * np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sigma_psf**2))

    # Small galaxies (extended Gaussians)
    for _ in range(n_galaxies):
        cx, cy = rng_local.uniform(10, size - 10, 2)
        sg = rng_local.uniform(2.0, 4.0)
        peak = rng_local.uniform(100, 500)
        yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
        img += peak * np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sg**2))

    # Poisson + read noise
    img = rng_local.poisson(img).astype(np.float64) + rng_local.normal(0, rn, img.shape)

    # Cosmic rays — sharp spikes (1-3 pixels)
    for _ in range(n_cr):
        cx, cy = rng_local.integers(2, size - 2), rng_local.integers(2, size - 2)
        peak = rng_local.uniform(800, 4000)
        sz = rng_local.choice([1, 2, 3])
        if sz == 1:
            img[cy, cx] += peak
            cr_mask[cy, cx] = True
        elif sz == 2:
            img[cy:cy+2, cx:cx+2] += peak / 2
            cr_mask[cy:cy+2, cx:cx+2] = True
        else:
            # Short streak (mu-on-like)
            for k in range(sz):
                img[cy, cx + k] += peak / sz
                cr_mask[cy, cx + k] = True

    return img, cr_mask


img, cr_mask = make_synthetic_field()
print(f'Image shape    : {img.shape}')
print(f'CR pixel count : {cr_mask.sum()}')
print(f'Mean count     : {img.mean():.1f} e-')
print(f'Max count      : {img.max():.1f} e-')

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(img, cmap='gray_r', vmin=80, vmax=300, origin='lower')
axes[0].set_title('Synthetic field (stars + galaxies + CRs)'); axes[0].axis('off')
axes[1].imshow(cr_mask, cmap='Reds', origin='lower')
axes[1].set_title(f'Ground-truth CR mask ({cr_mask.sum()} pixels)'); axes[1].axis('off')
plt.tight_layout(); plt.show()


---

## Part 2: Laplacian + 2× sub-sampling / Laplacian과 2배 서브샘플링

Eqs. 4–8: kernel, 2× upsample, Laplacian convolution, retain positive part, block average back.


In [ ]:
def laplacian_kernel() -> np.ndarray:
    """Eq. 4 — discrete Laplacian."""
    return 0.25 * np.array([[ 0, -1,  0],
                            [-1,  4, -1],
                            [ 0, -1,  0]], dtype=np.float64)


def upsample_2x(I: np.ndarray) -> np.ndarray:
    """Eq. 5 — 2x replicate sub-sample."""
    return np.repeat(np.repeat(I, 2, axis=0), 2, axis=1)


def block_avg_2x(I: np.ndarray) -> np.ndarray:
    """Eq. 8 — block average a 2x array back to original resolution."""
    h, w = I.shape
    return 0.25 * (I[0:h:2, 0:w:2] + I[0:h:2, 1:w:2]
                   + I[1:h:2, 0:w:2] + I[1:h:2, 1:w:2])


def lacosmic_laplacian_image(I: np.ndarray) -> np.ndarray:
    """Compute L^+ as in Eqs. 5-8."""
    I2 = upsample_2x(I)
    L2 = ndimage.convolve(I2, laplacian_kernel(), mode='nearest')
    L2_pos = np.maximum(L2, 0.0)
    return block_avg_2x(L2_pos)


L_pos = lacosmic_laplacian_image(img)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(img, cmap='gray_r', vmin=80, vmax=300, origin='lower'); axes[0].set_title('Input'); axes[0].axis('off')
axes[1].imshow(L_pos, cmap='hot', vmin=0, vmax=np.percentile(L_pos, 99.5), origin='lower')
axes[1].set_title('L+ (positive Laplacian, block averaged)'); axes[1].axis('off')
plt.tight_layout(); plt.show()


---

## Part 3: Noise model and significance image / 잡음 모델과 유의도 영상

Eq. 10: \$N = g^{-1}\sqrt{g\,(M_5\circ I) + \sigma_{\rm rn}^2}\$.
Eq. 11: \$S = \mathcal L^+ / (f_s N)\$, \$f_s = 2\$.


In [ ]:
def noise_model(I: np.ndarray, gain: float = 1.0, rn: float = 5.0) -> np.ndarray:
    """Eq. 10: N = (1/g) sqrt( g (M_5 o I) + sigma_rn^2 )."""
    M5 = ndimage.median_filter(I, size=5, mode='reflect')
    return np.sqrt(np.maximum(gain * M5, 0.0) + rn**2) / gain


def significance(I: np.ndarray, gain: float = 1.0, rn: float = 5.0) -> tuple[np.ndarray, np.ndarray]:
    """Compute (S, L^+) for the input image."""
    L_pos = lacosmic_laplacian_image(I)
    N = noise_model(I, gain=gain, rn=rn)
    S = L_pos / (2.0 * N)  # f_s = 2
    return S, L_pos


S, L_pos = significance(img)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
im0 = axes[0].imshow(S, cmap='hot', vmin=0, vmax=20, origin='lower')
axes[0].set_title(r'Significance $S = L^+/(2N)$'); axes[0].axis('off'); plt.colorbar(im0, ax=axes[0], fraction=0.046)
axes[1].imshow(S > 5, cmap='Reds', origin='lower'); axes[1].set_title(r'$S > 5\sigma$ candidates'); axes[1].axis('off')
plt.tight_layout(); plt.show()

# Subtract slow component (Eq. 13 sampling-flux removal)
S_minus_smooth = S - ndimage.median_filter(S, size=5)
print(f'5-sigma candidates total: {(S > 5).sum()}; after slow-removal: {(S_minus_smooth > 5).sum()}')


---

## Part 4: Fine-structure image and final criterion / 정밀 구조 영상과 최종 기준

Eq. 14: \$\mathcal F = (M_3\circ I) - ((M_3\circ I)\circ M_7)\$.
Decision: pixel is CR iff \$S' > \sigma_{\rm lim}\$ AND \$\mathcal L^+/\mathcal F > f_{\rm lim}\$.


In [ ]:
def fine_structure(I: np.ndarray) -> np.ndarray:
    """Eq. 14: F = (M_3 o I) - ((M_3 o I) o M_7)."""
    M3I = ndimage.median_filter(I, size=3, mode='reflect')
    M7M3I = ndimage.median_filter(M3I, size=7, mode='reflect')
    return M3I - M7M3I


def lacosmic_one_pass(
    I: np.ndarray,
    gain: float = 1.0,
    rn: float = 5.0,
    sigclip: float = 5.0,
    objlim: float = 2.0,
) -> np.ndarray:
    """Run one L.A.Cosmic pass, return the boolean CR mask."""
    S, L_pos = significance(I, gain=gain, rn=rn)
    S_minus = S - ndimage.median_filter(S, size=5)
    F_img = fine_structure(I)
    eps = 1e-12
    ratio = L_pos / (np.maximum(F_img, eps))
    mask = (S_minus > sigclip) & (ratio > objlim)
    return mask


# One-pass on synthetic image (well-sampled => objlim=2 default)
mask_pred = lacosmic_one_pass(img, gain=1.0, rn=5.0, sigclip=5.0, objlim=2.0)

# Detection statistics
TP = (mask_pred & cr_mask).sum()
FP = (mask_pred & ~cr_mask).sum()
FN = (~mask_pred & cr_mask).sum()
TN = (~mask_pred & ~cr_mask).sum()
print(f'TP = {TP}, FP = {FP}, FN = {FN}, TN = {TN}')
print(f'TPR (recall) = {TP / max(TP + FN, 1):.3f}')
print(f'FPR          = {FP / max(FP + TN, 1):.5f}')
print(f'Precision    = {TP / max(TP + FP, 1):.3f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cr_mask, cmap='Reds', origin='lower'); axes[0].set_title('Ground truth CR'); axes[0].axis('off')
axes[1].imshow(mask_pred, cmap='Blues', origin='lower'); axes[1].set_title('L.A.Cosmic detection'); axes[1].axis('off')
diff = np.zeros((*cr_mask.shape, 3))
diff[..., 0] = cr_mask & ~mask_pred  # missed (red)
diff[..., 2] = mask_pred & ~cr_mask  # false positive (blue)
diff[..., 1] = cr_mask & mask_pred   # correct (green)
axes[2].imshow(diff, origin='lower'); axes[2].set_title('Green=TP  Red=FN  Blue=FP'); axes[2].axis('off')
plt.tight_layout(); plt.show()


---

## Part 5: Iterative application for large CRs / 큰 CR을 위한 반복 적용

식별된 CR 픽셀을 주변 median으로 채우고 반복.


In [ ]:
def lacosmic(
    I: np.ndarray,
    gain: float = 1.0,
    rn: float = 5.0,
    sigclip: float = 5.0,
    objlim: float = 2.0,
    n_iter: int = 4,
    verbose: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """Iterate L.A.Cosmic, replacing each detected CR with the median of its 5x5 neighbourhood.

    Returns:
        (cleaned_image, cumulative_cr_mask).
    """
    I_curr = I.copy()
    mask_total = np.zeros_like(I, dtype=bool)
    for k in range(n_iter):
        m = lacosmic_one_pass(I_curr, gain=gain, rn=rn, sigclip=sigclip, objlim=objlim)
        m &= ~mask_total  # only newly found pixels
        if verbose:
            print(f'  iter {k+1}: {m.sum()} new CR pixels')
        if m.sum() == 0:
            break
        median_local = ndimage.median_filter(I_curr, size=5)
        I_curr = np.where(m, median_local, I_curr)
        mask_total |= m
    return I_curr, mask_total


cleaned, mask_total = lacosmic(img, n_iter=4)
TP = (mask_total & cr_mask).sum(); FP = (mask_total & ~cr_mask).sum()
FN = (~mask_total & cr_mask).sum(); TN = (~mask_total & ~cr_mask).sum()
print(f'\nFinal: TP={TP}, FP={FP}, FN={FN}')
print(f'  TPR = {TP / max(TP + FN, 1):.3f},  FPR = {FP / max(FP + TN, 1):.5f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(img,     cmap='gray_r', vmin=80, vmax=300, origin='lower'); axes[0].set_title('Before'); axes[0].axis('off')
axes[1].imshow(cleaned, cmap='gray_r', vmin=80, vmax=300, origin='lower'); axes[1].set_title('After L.A.Cosmic (4 iter)'); axes[1].axis('off')
plt.tight_layout(); plt.show()


---

## Part 6: Compare against `astroscrappy` if available / 라이브러리 비교

`astroscrappy` (pip install astroscrappy) provides Curtis McCully's optimised C-extension implementation. We compare results if installed.


In [ ]:
try:
    import astroscrappy as ast
    cr_mask_ast, cleaned_ast = ast.detect_cosmics(img, gain=1.0, readnoise=5.0,
                                                  sigclip=5.0, objlim=2.0, niter=4)
    TP = (cr_mask_ast & cr_mask).sum()
    FP = (cr_mask_ast & ~cr_mask).sum()
    FN = (~cr_mask_ast & cr_mask).sum()
    print(f'astroscrappy: TP={TP} FP={FP} FN={FN}')
    print(f'   TPR={TP/(TP+FN):.3f}  FPR={FP/(FP+(~cr_mask).sum()-FP):.5f}')

    overlap = (mask_total & cr_mask_ast).sum() / max(mask_total.sum(), 1)
    print(f'Pixel agreement (ours ∩ astroscrappy)/ours = {overlap:.3f}')
except ImportError:
    print('astroscrappy not installed; skipping cross-check.')
    print('Install with: pip install astroscrappy')


---

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 본 노트북 |
|---|---|---|
| Discrete Laplacian | Eq. 4 | `laplacian_kernel()` |
| 2× sub-sample | Eq. 5 | `upsample_2x()` |
| Block average | Eq. 8 | `block_avg_2x()` |
| Noise model (Poisson + RN) | Eq. 10 | `noise_model()` |
| Significance map | Eq. 11 | `significance()` |
| Fine-structure image | Eq. 14 | `fine_structure()` |
| One-pass detection | §3.1-3.2 | `lacosmic_one_pass()` |
| Iterative peeling | §3.3 | `lacosmic()` |
| Library reference | C-extension | `astroscrappy.detect_cosmics()` |

### Connections to other papers / 다른 논문과의 연결
- **Paper #24 deepCR**: deep-learning successor that uses L.A.Cosmic as the standard ROC baseline.
- **Paper #1 Donoho-Johnstone**: similar template "transform → threshold by noise model → invert" applied to denoising rather than CR rejection.
- **Marr-Hildreth (1980)**: theoretical foundation for zero-crossing edge detection.

### Take-away / 결론
The Laplacian of the 2x sub-sampled image cleanly separates the *sharp* CR edges from the *smoothly* sampled stars — the synthetic-field experiment recovers the bulk of CR pixels at low false-positive rate using only `numpy + scipy`. The four ingredients (sub-sample, Laplacian, fine-structure ratio, iterate) each play a non-redundant role.

2× 서브샘플링된 영상의 Laplacian은 *날카로운* CR 에지와 *매끄럽게 표본화*된 별을 깔끔히 분리한다. NumPy + SciPy만으로도 합성 영상의 대부분 CR을 낮은 false positive rate로 검출. 네 요소(서브샘플 / Laplacian / fine-structure 비율 / 반복)는 각각 고유한 역할을 한다.
